# 02b · Video Acquisition Recovery

**Project:** Isolated word-level ASL recognition · **Dataset:** WLASL-100 · **Follows:** Notebook 02 (left untouched)

## What this notebook does
1. Builds a **fresh instance-level audit** from the files actually on disk (not from earlier summaries).
2. Classifies every instance and **explains previous failures** from the yt-dlp log, the official downloader logs and this notebook's own attempt log.
3. Writes a **recovery-priority report before any download starts**.
4. Recovers sources incrementally: **extract clips from sources already on disk first (no network)**, then YouTube, then non-YouTube URLs. Bounded concurrency, timeouts, backoff, retry budgets, runtime cap, resumable.
5. Validates every recovered clip and writes the recovery artifacts plus a consolidated audit, then runs integrity checks.

## Hard constraints (enforced by code, not just by convention)
- Official WLASL metadata, `WLASL100_filtered.json`, the frozen split CSVs and signer assignments are **opened read-only**; their SHA-256 hashes are recorded in a baseline file and re-checked at the end.
- Existing valid clips are never overwritten, moved or re-encoded. Their size and SHA-256 are recorded and re-checked.
- No randomness is used in the experiment; backoff jitter uses a private `random.Random()` and never touches the global seed.
- No authentication, paywall or access-restriction bypass. Private, age-gated, geo-blocked and 401/403 sources are recorded and **not** retried.

**Source downloads and usable instance clips are always counted separately.** A *usable instance* is a valid file `clips/<video_id>.mp4`.

## 1. Configuration

In [1]:
from pathlib import Path
import os, re, sys, json, time, shutil, hashlib, socket, threading, subprocess, random
import urllib.request, urllib.error
from urllib.parse import urlparse, parse_qs
from datetime import datetime, timezone, timedelta
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import cv2
from IPython.display import display

PROJECT_DIR = None  # e.g. Path.home() / "sign-language-recognition"; None = auto-detect

# ---- what to run -------------------------------------------------------------
RUN_EXTRACTION       = True   # crop clips from sources ALREADY on disk (no network)
RUN_YOUTUBE          = True   # network: yt-dlp for eligible sources only
RUN_NON_YOUTUBE      = True   # network: HTTP(S) for eligible URLs only
RUN_SWF_CONVERSION   = True   # needs ffmpeg on PATH; validated, never trusted blindly

# ---- resource limits (MacBook Air friendly) -------------------------------------
MAX_RUNTIME_MIN      = 120    # wall-clock cap for the whole network phase; None = unlimited
YT_WORKERS           = 2
HTTP_WORKERS         = 4
PER_HOST_MAX         = 2      # concurrent requests per host
HOST_MIN_DELAY_SEC   = 1.0    # polite spacing between requests to one host
EXTRACT_WORKERS      = 2
cv2.setNumThreads(2)

# ---- timeouts / retries ---------------------------------------------------------
HTTP_TIMEOUT_SEC         = 20      # socket timeout (connect + each read)
HTTP_MAX_SECONDS         = 180     # hard cap on one download
MAX_DOWNLOAD_MB          = 300
YT_SOCKET_TIMEOUT_SEC    = 30
MAX_HEIGHT               = 1080    # yt-dlp format ceiling (prefers H.264 for OpenCV)
IN_RUN_TRIES             = 2       # tries within one run for temporary errors
BACKOFF_BASE_SEC         = 5.0     # sleep = base * 2**k + jitter
MAX_TEMP_ATTEMPTS_TOTAL  = 3       # across ALL runs, per source, for temporary failures
COOLDOWN_HOURS           = 6       # do not re-hit a temporarily failed source sooner than this
HOST_BREAKER             = 8       # consecutive failures with zero successes -> skip host this run
COOKIES_FROM_BROWSER     = None    # e.g. ("safari",) to use YOUR OWN logged-in session for bot-check errors only
LIMIT_NEW_YT_SOURCES     = None    # debugging aid; None = no limit
LIMIT_NEW_URLS           = None

# ---- validation limits ----------------------------------------------------------
MIN_DIM, MAX_DIM         = 32, 8192
MIN_FPS, MAX_FPS         = 5.0, 240.0
MIN_CLIP_FRAMES          = 5
MAX_CLIP_SEC             = 60.0   # longer "clips" are rejected and reported as implausible_clip_duration
MAX_SOURCE_SEC           = 4 * 3600

UA = ("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 "
      "(KHTML, like Gecko) Version/17.0 Safari/605.1.15")
RNG = random.Random()   # private; never touches the experiment seed
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
def utc_now(): return datetime.now(timezone.utc).isoformat()

## 2. Project paths and frozen inputs (read-only)

In [2]:
def find_project_root(start):
    for c in [start, *start.parents, Path.home() / "sign-language-recognition"]:
        if (c / "WLASL" / "start_kit" / "WLASL_v0.3.json").exists():
            return c.resolve()
    return None

project_dir = Path(PROJECT_DIR).expanduser().resolve() if PROJECT_DIR else find_project_root(Path.cwd())
if project_dir is None:
    raise FileNotFoundError("Cannot locate WLASL/start_kit/WLASL_v0.3.json. Set PROJECT_DIR.")

official_json   = project_dir / "WLASL" / "start_kit" / "WLASL_v0.3.json"
processed_dir   = project_dir / "data" / "processed" / "WLASL100"
source_dir      = project_dir / "data" / "raw" / "youtube_sources"
non_yt_dir      = project_dir / "data" / "raw" / "non_youtube_instances"
clip_dir        = processed_dir / "clips"
staging_dir     = project_dir / "data" / "raw" / "recovery_staging"
quarantine_dir  = project_dir / "data" / "raw" / "recovery_quarantine"
for d in (processed_dir, source_dir, non_yt_dir, clip_dir, staging_dir, quarantine_dir):
    d.mkdir(parents=True, exist_ok=True)

filtered_json = processed_dir / "WLASL100_filtered.json"
split_files = {s: processed_dir / f"{s}_metadata.csv" for s in ("train", "val", "test")}
for p in (official_json, filtered_json, *split_files.values()):
    if not p.exists():
        raise FileNotFoundError(f"Required frozen input missing: {p}")

OUT = {n: processed_dir / n for n in [
    "recovery_attempt_log.csv", "recovery_instance_status.csv", "recovery_source_status.csv",
    "recovery_split_availability.csv", "recovery_class_coverage.csv", "recovery_summary.json",
    "recovery_priority_report.csv", "recovery_baseline.json", "consolidated_acquisition_audit.csv"]}
print("Project root:", project_dir)
print("Run id:", RUN_ID)

Project root: /Users/sahilchauhan/sign-language-recognition
Run id: 20261104T120218Z


In [3]:
def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

with filtered_json.open() as f:
    filtered_data = json.load(f)
records = []
for entry in filtered_data:
    for inst in entry["instances"]:
        rec = {"gloss": entry["gloss"], **inst}
        rec["video_id"] = str(rec["video_id"]).zfill(5)
        records.append(rec)
instances_df = pd.DataFrame(records)
# The official WLASL JSON carries its own `split` field; the FROZEN split lives in the CSVs. Keep the former under another name.
instances_df = instances_df.rename(columns={"split": "official_wlasl_split"})
assert len(instances_df) == 2038 and instances_df["video_id"].is_unique
assert instances_df["gloss"].nunique() == 100 and instances_df["signer_id"].nunique() == 97

parts = []
for s, p in split_files.items():
    part = pd.read_csv(p, dtype={"video_id": str})
    part["video_id"] = part["video_id"].str.zfill(5)
    part["split"] = s
    parts.append(part[["video_id", "gloss", "signer_id", "split"]])
split_df = pd.concat(parts, ignore_index=True)
assert not split_df["video_id"].duplicated().any(), "video_id in more than one split"
assert set(split_df["video_id"]) == set(instances_df["video_id"]), "split CSVs do not cover the 2,038 instances"

inst = instances_df.merge(split_df[["video_id", "split", "signer_id"]].rename(columns={"signer_id": "split_signer_id"}),
                          on="video_id", how="left", validate="1:1")
assert (inst["signer_id"].astype(int) == inst["split_signer_id"].astype(int)).all(), "signer mismatch between metadata and frozen splits"
inst = inst.drop(columns="split_signer_id")
# signer independence of the frozen splits (checked, never changed)
sig_split = inst.groupby("signer_id")["split"].nunique()
print("Signers appearing in >1 split (frozen design):", int((sig_split > 1).sum()))

def extract_youtube_id(url):
    if not isinstance(url, str) or not url.strip(): return None
    u = urlparse(url); host = u.netloc.lower().split(":")[0]
    if host == "youtu.be" or host.endswith(".youtu.be"):
        return u.path.strip("/").split("/")[0] or None
    if host == "youtube.com" or host.endswith(".youtube.com"):
        parts_ = u.path.strip("/").split("/")
        if parts_ and parts_[0] in {"shorts", "embed", "live"} and len(parts_) > 1: return parts_[1]
        return parse_qs(u.query).get("v", [None])[0]
    return None

inst["youtube_id"] = inst["url"].apply(extract_youtube_id)
inst["is_youtube"] = inst["youtube_id"].notna()
inst["host"] = inst["url"].apply(lambda u: urlparse(u).netloc.lower())
inst["source_key"] = np.where(inst["is_youtube"], inst["youtube_id"],
                              "url:" + inst["url"].apply(lambda u: hashlib.sha1(u.encode()).hexdigest()[:12]))
print(f"YouTube instances {inst['is_youtube'].sum()} / sources {inst.loc[inst.is_youtube,'youtube_id'].nunique()};"
      f" non-YouTube instances {(~inst['is_youtube']).sum()} / unique URLs {inst.loc[~inst.is_youtube,'url'].nunique()}")
display(inst.loc[~inst.is_youtube, "host"].value_counts().rename("instances").to_frame().T)

Signers appearing in >1 split (frozen design): 0
YouTube instances 685 / sources 572; non-YouTube instances 1353 / unique URLs 1341


host,www.signingsavvy.com,signstock.blob.core.windows.net,www.handspeak.com,www.aslpro.com,media.asldeafined.com,www.aslsearch.com,aslsignbank.haskins.yale.edu,s3-us-west-1.amazonaws.com,media.spreadthesign.com,aslbricks.org,elementalaslconcepts.weebly.com
instances,225,196,174,162,131,114,108,99,91,45,8


## 3. Validation utilities

`audit_video` is stricter than Notebook 02's: besides OpenCV open + decoding at 0 / 50 / 95 %, it checks dimensions, FPS and duration plausibility. For newly extracted clips it also **decodes every frame** and compares the count with the expected frame range. Results are cached by `(path, size, mtime)` so unchanged files are never re-decoded.

In [4]:
VIDEO_EXTS = (".mp4", ".mov", ".mkv", ".avi", ".webm")
_audit_cache, _audit_lock = {}, threading.Lock()

def audit_video(path, kind="source", expected_frames=None, full_decode=False):
    path = Path(path)
    r = {"path": str(path), "exists": path.is_file(), "valid": False, "width": None, "height": None,
         "fps": None, "frame_count": None, "duration_sec": None, "size_mb": 0.0, "error": None}
    if not r["exists"]:
        r["error"] = "file_not_found"; return r
    size = path.stat().st_size
    r["size_mb"] = round(size / 1048576, 3)
    if size == 0:
        r["error"] = "zero_size"; return r
    cap = cv2.VideoCapture(str(path))
    try:
        if not cap.isOpened():
            r["error"] = "opencv_open_failed"; return r
        w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps, n = float(cap.get(cv2.CAP_PROP_FPS)), int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        r.update(width=w, height=h, fps=fps, frame_count=n, duration_sec=(n / fps if fps > 0 else None))
        if w <= 0 or h <= 0 or fps <= 0 or n <= 0:
            r["error"] = "invalid_video_metadata"; return r
        if not (MIN_DIM <= w <= MAX_DIM and MIN_DIM <= h <= MAX_DIM):
            r["error"] = "implausible_dimensions"; return r
        if not (MIN_FPS <= fps <= MAX_FPS):
            r["error"] = "implausible_fps"; return r
        dur = n / fps
        if kind == "clip":
            if n < MIN_CLIP_FRAMES or dur > MAX_CLIP_SEC:
                r["error"] = "implausible_clip_duration"; return r
        elif dur < 0.2 or dur > MAX_SOURCE_SEC:
            r["error"] = "implausible_source_duration"; return r
        if expected_frames is not None and abs(n - expected_frames) > 2:
            r["error"] = "frame_count_mismatch"; return r
        ok_all = []
        for frac in (0.0, 0.5, 0.95):
            cap.set(cv2.CAP_PROP_POS_FRAMES, min(int(n * frac), n - 1))
            ok, fr = cap.read()
            ok_all.append(bool(ok and fr is not None and fr.size))
        if not all(ok_all):
            r["error"] = "frame_decode_failed"; return r
        if full_decode:
            cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
            cnt = 0
            while True:
                ok, fr = cap.read()
                if not ok: break
                if fr is None or not fr.size:
                    r["error"] = "frame_decode_failed"; return r
                cnt += 1
            if abs(cnt - n) > 2 or (expected_frames is not None and abs(cnt - expected_frames) > 2):
                r["error"] = "decoded_frame_count_mismatch"; return r
        r["valid"] = True
        return r
    finally:
        cap.release()

def cached_audit(path, kind="source"):
    path = Path(path)
    if not path.is_file():
        return audit_video(path, kind)
    st = path.stat()
    key = (str(path), st.st_size, st.st_mtime_ns, kind)
    with _audit_lock:
        if key in _audit_cache: return _audit_cache[key]
    res = audit_video(path, kind)
    with _audit_lock:
        _audit_cache[key] = res
    return res

def find_video_file(folder, stem, extensions=VIDEO_EXTS):
    for ext in extensions:
        p = Path(folder) / f"{stem}{ext}"
        if p.is_file(): return p
    return None

## 4. Failure classification

Every error text (yt-dlp, urllib, HTTP) is mapped to a **category** and a **permanence**:
- `permanent` – will not be retried (private, removed, account terminated, age-gated, geo-blocked from your location, members-only, 401/403, 404/410, HTML instead of video).
- `temporary` – retried with backoff, bounded by `MAX_TEMP_ATTEMPTS_TOTAL` across runs and a cooldown.

In [5]:
PERMANENT_PATTERNS = [
    ("private video", "private"), ("video is private", "private"),
    ("account associated with this video has been terminated", "account_terminated"),
    ("has been removed", "removed"), ("removed by the uploader", "removed"),
    ("copyright", "copyright_blocked"), ("violating youtube's terms", "removed"),
    ("this video is unavailable", "unavailable"), ("video unavailable", "unavailable"),
    ("video is not available", "unavailable"), ("no longer available", "unavailable"),
    ("does not exist", "unavailable"),
    ("confirm your age", "age_restricted"), ("age-restricted", "age_restricted"), ("age restricted", "age_restricted"),
    ("not available in your country", "geo_blocked"), ("blocked it in your country", "geo_blocked"),
    ("members-only", "members_only"), ("join this channel", "members_only"),
    ("live event will begin", "not_a_vod"), ("premieres in", "not_a_vod"),
]
TEMPORARY_PATTERNS = [
    ("http error 429", "rate_limited"), ("too many requests", "rate_limited"),
    ("not a bot", "bot_check"), ("po token", "bot_check"),
    ("timed out", "timeout"), ("timeout", "timeout"),
    ("name or service not known", "dns"), ("nodename nor servname", "dns"), ("temporary failure in name resolution", "dns"),
    ("getaddrinfo", "dns"), ("network is unreachable", "network"), ("connection reset", "network"),
    ("connection refused", "network"), ("connection aborted", "network"), ("remote end closed", "network"),
    ("ssl", "ssl_error"), ("certificate", "ssl_error"),
    ("http error 5", "server_error"), ("requested format is not available", "format_unavailable"),
    ("incomplete", "incomplete_download"),
]
def classify_text(text):
    t = (text or "").lower()
    for pat, cat in PERMANENT_PATTERNS:
        if pat in t: return cat, "permanent"
    for pat, cat in TEMPORARY_PATTERNS:
        if pat in t: return cat, "temporary"
    return "other_error", "temporary"

def classify_http(code):
    if code in (404, 410): return "http_not_found", "permanent"
    if code in (401, 403, 451): return "http_access_denied", "permanent"   # never bypass
    if code in (408, 425, 429): return "http_rate_or_timeout", "temporary"
    if 500 <= code < 600: return "http_server_error", "temporary"
    if 400 <= code < 500: return "http_client_error", "permanent"
    return "http_other", "temporary"

## 5. Attempt log (resumable memory) and legacy logs

`recovery_attempt_log.csv` is **append-only**. It is the memory that makes reruns cheap: permanent failures are never retried, temporary ones respect the retry budget and cooldown. Legacy evidence is read-only: `youtube_download_log.json` (Notebook 02) and any `download_*.log` written by the official downloader.

In [6]:
ATTEMPT_COLS = ["timestamp_utc", "run_id", "kind", "source_key", "video_ids", "status", "category",
                "permanence", "http_status", "detail", "bytes", "seconds"]
_log_lock = threading.Lock()
def log_attempt(**row):
    row = {c: row.get(c) for c in ATTEMPT_COLS} | {"timestamp_utc": utc_now(), "run_id": RUN_ID}
    with _log_lock:
        new = not OUT["recovery_attempt_log.csv"].exists()
        pd.DataFrame([row], columns=ATTEMPT_COLS).to_csv(OUT["recovery_attempt_log.csv"], mode="a", header=new, index=False)

def load_attempts():
    p = OUT["recovery_attempt_log.csv"]
    return pd.read_csv(p, dtype=str) if p.exists() else pd.DataFrame(columns=ATTEMPT_COLS)

# --- legacy YouTube log (Notebook 02) ---
legacy_yt = {}
p = source_dir / "youtube_download_log.json"
if p.exists():
    for k, v in json.loads(p.read_text()).items():
        st = str(v.get("status", ""))
        if st.startswith("downloaded_and_validated"):
            continue
        cat, perm = classify_text(st)
        legacy_yt[k] = {"status": st[:300], "category": cat, "permanence": perm, "ts": v.get("timestamp_utc")}
print("Legacy yt-dlp failures:", len(legacy_yt), dict(Counter(v["category"] for v in legacy_yt.values())))

# --- official downloader logs (non-YouTube) ---
legacy_official = {}   # video_id -> "exists" | "unsuccessful"
pat_vid = re.compile(r"gloss: .*?, video: (\d+)\.")
pat_bad = re.compile(r"Unsuccessful downloading - video (\d+)")
pat_ok = re.compile(r"^(\d+) exists at ")
log_files = [q for q in project_dir.rglob("download_*.log") if ".git" not in q.parts]
for lf in sorted(log_files):
    try:
        for line in lf.read_text(errors="ignore").splitlines():
            m = pat_bad.search(line)
            if m: legacy_official[m.group(1).zfill(5)] = "unsuccessful"; continue
            m = pat_ok.match(line)
            if m and legacy_official.get(m.group(1).zfill(5)) != "unsuccessful": legacy_official[m.group(1).zfill(5)] = "exists"
    except OSError:
        pass
print(f"Official downloader logs found: {len(log_files)}; instances with a logged failure: "
      f"{sum(v=='unsuccessful' for v in legacy_official.values())}")
print("NOTE: the official downloader swallows the exception text, so the reason for those failures is unknown until rechecked.")

Legacy yt-dlp failures: 60 {'unavailable': 33, 'private': 27}
Official downloader logs found: 1; instances with a logged failure: 4
NOTE: the official downloader swallows the exception text, so the reason for those failures is unknown until rechecked.


## 6. Frozen-state baseline (written once, never overwritten)

Recorded before anything is changed: hashes of every frozen input, and size + SHA-256 of every currently valid clip. Reruns reuse the same baseline, so improvements are always measured against the original state.

In [7]:
baseline_path = OUT["recovery_baseline.json"]
if baseline_path.exists():
    baseline = json.loads(baseline_path.read_text())
    print("Reusing baseline created", baseline["created_utc"])
else:
    clips0 = {}
    for r in inst.itertuples(index=False):
        cp = clip_dir / f"{r.video_id}.mp4"
        if cp.is_file():
            a = audit_video(cp, "clip")
            if a["valid"]:
                clips0[r.video_id] = {"size": cp.stat().st_size, "sha256": sha256_file(cp)}
    baseline = {
        "created_utc": utc_now(),
        "frozen_hashes": {str(q.relative_to(project_dir)): sha256_file(q) for q in (official_json, filtered_json, *split_files.values())},
        "baseline_valid_clips": clips0,
        "baseline_valid_youtube_sources": sorted(
            sid for sid in inst.loc[inst.is_youtube, "youtube_id"].unique()
            if (fp := find_video_file(source_dir, sid)) and audit_video(fp)["valid"]),
    }
    baseline_path.write_text(json.dumps(baseline, indent=2))
    print("Baseline written.")
baseline_valid_clips = set(baseline["baseline_valid_clips"])
print("Baseline valid instance clips:", len(baseline_valid_clips),
      "| baseline valid YouTube source files:", len(baseline["baseline_valid_youtube_sources"]))

Baseline written.
Baseline valid instance clips: 457 | baseline valid YouTube source files: 340


## 7. Fresh instance-level acquisition audit

Classes (first match wins):
1. `already_acquired_valid` – a valid `clips/<id>.mp4` exists.
2. `source_valid_clip_pending` – the source video is on disk and valid, but the instance clip was never extracted. **Recoverable offline.**
3. `downloaded_but_invalid` – a file exists but fails validation.
4. `missing_or_permanently_inaccessible` – a permanent failure is on record.
5. `previously_attempted_failed` – attempted before, failure not proven permanent.
6. `never_attempted`.

In [8]:
def build_audit():
    attempts = load_attempts()
    last_by_key, temp_n, last_temp_ts = {}, Counter(), {}
    for r in attempts.itertuples(index=False):
        if r.kind in ("source_youtube", "source_url") and r.status != "success":
            last_by_key[r.source_key] = r
            if r.permanence == "temporary":
                temp_n[r.source_key] += 1; last_temp_ts[r.source_key] = r.timestamp_utc
    rows = []
    for r in inst.itertuples(index=False):
        vid = r.video_id
        clip_p = clip_dir / f"{vid}.mp4"
        clip_a = cached_audit(clip_p, "clip") if clip_p.is_file() else None
        if r.is_youtube:
            sp = find_video_file(source_dir, r.youtube_id)
        else:
            sp = find_video_file(non_yt_dir, vid)
            swf = non_yt_dir / f"{vid}.swf"
        s_a = cached_audit(sp, "source") if sp else None
        has_swf = (not r.is_youtube) and swf.is_file()
        perm = temp = None; cat = reason = None; hist = "none"
        key = r.source_key
        lr = last_by_key.get(key)
        if lr is not None:
            hist, cat, perm, reason = "recovery_log", lr.category, lr.permanence, lr.detail
        elif r.is_youtube and key in legacy_yt:
            hist, cat, perm, reason = "notebook02_log", legacy_yt[key]["category"], legacy_yt[key]["permanence"], legacy_yt[key]["status"]
        elif (not r.is_youtube) and legacy_official.get(vid) == "unsuccessful":
            hist, cat, perm, reason = "official_downloader_log", "unknown_reason", "unknown", "exception text not logged by official downloader"
        if clip_a is not None and clip_a["valid"]:
            ac = "already_acquired_valid"
        elif s_a is not None and s_a["valid"]:
            ac = "source_valid_clip_pending"
        elif (s_a is not None and s_a["exists"]) or (clip_a is not None and clip_a["exists"]) or has_swf:
            ac = "downloaded_but_invalid"
            if has_swf and not (s_a and s_a["exists"]): cat = cat or "swf_needs_conversion"
            if s_a is not None and s_a["exists"] and s_a["error"]: cat = s_a["error"]
        elif perm == "permanent":
            ac = "missing_or_permanently_inaccessible"
        elif hist != "none":
            ac = "previously_attempted_failed"
        else:
            ac = "never_attempted"
        if ac in ("already_acquired_valid", "source_valid_clip_pending"):
            cat = perm = None; reason = ""; hist = hist if hist == "none" else hist + "(resolved)"
        rows.append({"video_id": vid, "gloss": r.gloss, "signer_id": r.signer_id, "split": r.split,
                     "is_youtube": r.is_youtube, "source_key": key, "host": r.host, "url": r.url,
                     "frame_start": r.frame_start, "frame_end": r.frame_end, "meta_fps": r.fps,
                     "acq_class": ac, "history_source": hist, "fail_category": cat, "permanence": perm,
                     "fail_detail": (reason or "")[:300],
                     "temp_attempts": int(temp_n.get(key, 0)), "last_temp_attempt_utc": last_temp_ts.get(key),
                     "source_path": str(sp) if sp else None, "source_valid": bool(s_a and s_a["valid"]),
                     "clip_valid": bool(clip_a and clip_a["valid"]),
                     "clip_error": clip_a["error"] if clip_a else "clip_not_found"})
    return pd.DataFrame(rows)

audit0 = build_audit()
print("Instance acquisition classes (fresh audit):")
display(audit0.groupby(["acq_class"]).size().rename("instances").to_frame()
        .join(audit0.pivot_table(index="acq_class", columns="split", values="video_id", aggfunc="count", fill_value=0)))
print("\nPrevious failure reasons (where known):")
display(audit0[audit0["fail_category"].notna()].groupby(["is_youtube", "fail_category", "permanence"], dropna=False).size().rename("instances").to_frame())
print("Source-backed (valid source or valid clip) vs. usable clips (valid clip file):")
print("  source-backed instances:", int((audit0.clip_valid | audit0.source_valid).sum()),
      "| usable instance clips:", int(audit0.clip_valid.sum()))

Instance acquisition classes (fresh audit):


,instances,test,train,val
acq_class,,,,
already_acquired_valid,457,244,182,31
missing_or_permanently_inaccessible,60,11,21,28
never_attempted,1414,341,740,333
previously_attempted_failed,3,1,0,2
source_valid_clip_pending,104,14,60,30



Previous failure reasons (where known):


instances
is_youtube fail_category  permanence           
False      unknown_reason unknown             3
True       private        permanent          27
           unavailable    permanent          33

Source-backed (valid source or valid clip) vs. usable clips (valid clip file):
  source-backed instances: 561 | usable instance clips: 457


## 8. Recovery-priority report (produced **before** any download)

Score per recoverable instance = split weight (train 100 / val 40 / test 10) + class-scarcity bonus (+50 if the class has no usable-or-pending example in that split, +20 if fewer than 3). Sources inherit the **sum** of their instances' scores, so a YouTube video backing several scarce training instances ranks first. Permanently unavailable sources get no score and are never queued.

In [9]:
SPLIT_W = {"train": 100, "val": 40, "test": 10}
def cooldown_ok(ts):
    if not isinstance(ts, str) or not ts: return True
    return datetime.now(timezone.utc) - datetime.fromisoformat(ts) >= timedelta(hours=COOLDOWN_HOURS)

def build_priority(audit):
    have = audit[audit.clip_valid | audit.source_valid].groupby(["split", "gloss"]).size()
    def bonus(r):
        n = have.get((r.split, r.gloss), 0)
        return 50 if n == 0 else (20 if n < 3 else 0)
    cand = audit[audit.acq_class.isin(["never_attempted", "previously_attempted_failed", "downloaded_but_invalid"])].copy()
    cand["score"] = cand["split"].map(SPLIT_W) + cand.apply(bonus, axis=1)
    def eligible(r):
        if r.permanence == "permanent": return False, "permanent_failure_on_record"
        if r.temp_attempts >= MAX_TEMP_ATTEMPTS_TOTAL: return False, "retry_budget_exhausted"
        if not cooldown_ok(r.last_temp_attempt_utc): return False, "cooling_down"
        return True, "eligible"
    if len(cand):
        el = cand.apply(lambda r: pd.Series(eligible(r), index=["eligible", "eligibility_reason"]), axis=1)
        cand = pd.concat([cand, el], axis=1)
    else:
        cand["eligible"] = pd.Series(dtype=bool); cand["eligibility_reason"] = pd.Series(dtype=str)
    src = (cand[cand.eligible].groupby(["source_key", "is_youtube", "host"])
           .agg(instances=("video_id", "count"), score=("score", "sum"),
                train=("split", lambda s: (s == "train").sum()), val=("split", lambda s: (s == "val").sum()),
                test=("split", lambda s: (s == "test").sum()), glosses=("gloss", lambda s: ", ".join(sorted(set(s))[:6])),
                fail_category=("fail_category", "first"))
           .reset_index().sort_values(["score", "instances"], ascending=False).reset_index(drop=True))
    src["rank"] = src.index + 1
    return cand, src

cand0, priority0 = build_priority(audit0)
priority0.to_csv(OUT["recovery_priority_report.csv"], index=False)

pend = audit0[audit0.acq_class == "source_valid_clip_pending"]
print("=== RECOVERY PRIORITY REPORT ===")
print(f"Step A (no network): extract clips from sources already on disk -> up to {len(pend)} instances "
      f"({pend.split.value_counts().to_dict()})")
print(f"Step B: YouTube sources eligible: {int(priority0.is_youtube.sum())} "
      f"({int(priority0[priority0.is_youtube].instances.sum())} instances)")
print(f"Step C: non-YouTube URLs eligible: {int((~priority0.is_youtube).sum())} "
      f"({int(priority0[~priority0.is_youtube].instances.sum())} instances)")
skipped = cand0[~cand0.eligible].groupby("eligibility_reason").size()
print("Skipped by policy:", skipped.to_dict())
print("\nTop 15 sources:")
display(priority0.head(15))
print("Eligible non-YouTube URLs by host:")
display(priority0[~priority0.is_youtube].groupby("host").agg(urls=("source_key", "count"), instances=("instances", "sum")).sort_values("instances", ascending=False))

=== RECOVERY PRIORITY REPORT ===
Step A (no network): extract clips from sources already on disk -> up to 104 instances ({'train': 60, 'val': 30, 'test': 14})
Step B: YouTube sources eligible: 172 (172 instances)
Step C: non-YouTube URLs eligible: 1233 (1245 instances)
Skipped by policy: {}

Top 15 sources:


,source_key,is_youtube,host,instances,score,train,val,test,glosses,fail_category,rank
0,url:60d25ca2feaf,False,www.aslsearch.com,2,180,0,2,0,orange,NaN,1
1,url:ccfb0cd8e761,False,www.aslsearch.com,2,180,0,2,0,later,NaN,2
2,ZxOqr6aCVLc,True,www.youtube.com,1,150,1,0,0,thanksgiving,NaN,3
3,pupBwmtznhI,True,www.youtube.com,1,150,1,0,0,medicine,NaN,4
4,qMgcmGOiNmw,True,www.youtube.com,1,150,1,0,0,give,NaN,5
5,rDAGt54vEsI,True,www.youtube.com,1,150,1,0,0,same,NaN,6
6,url:003b0156c776,False,www.handspeak.com,1,150,1,0,0,same,NaN,7
7,url:07a2cee089ce,False,www.handspeak.com,1,150,1,0,0,same,NaN,8
8,url:080f5d7a9c9d,False,media.asldeafined.com,1,150,1,0,0,medicine,NaN,9
9,url:0a1b9ed5a075,False,www.handspeak.com,1,150,1,0,0,thanksgiving,NaN,10


Eligible non-YouTube URLs by host:


,urls,instances
host,,
www.signingsavvy.com,215,225
www.handspeak.com,174,174
www.aslpro.com,162,162
signstock.blob.core.windows.net,157,157
www.aslsearch.com,112,114
aslsignbank.haskins.yale.edu,108,108
media.asldeafined.com,103,103
media.spreadthesign.com,78,78
s3-us-west-1.amazonaws.com,77,77


## 9. Proposed strategy
1. **Extract first.** Sources that are already valid on disk but have no instance clip need only local, cheap work. This is the largest and safest gain.
2. **YouTube:** one yt-dlp download per unique video id, highest priority first, format fallback ladder (H.264 ≤ `MAX_HEIGHT` → any ≤ `MAX_HEIGHT` → best), socket timeout, backoff. Private / unavailable / removed / age-gated / geo-blocked results are recorded as permanent and never retried — Notebook 02's `RETRY_PRIVATE_OR_UNAVAILABLE=True` spent its whole 20-source batch on exactly these.
3. **Non-YouTube:** one request per unique URL with content sniffing (magic bytes), so HTML error pages and login pages are never saved as video. Per-host throttling and a circuit breaker stop hammering dead hosts. `.swf` is converted with ffmpeg only if available, and validated.
4. **Validate, then publish**: everything downloads into `recovery_staging/`, is validated, and only then moved into the existing raw directories. Nothing existing is overwritten.

## 10. Generic bounded runner

In [10]:
DEADLINE = (time.time() + MAX_RUNTIME_MIN * 60) if MAX_RUNTIME_MIN else None
def out_of_time(): return DEADLINE is not None and time.time() > DEADLINE

def run_bounded(tasks, fn, workers, label):
    """Submit lazily so the runtime cap stops *new* work; in-flight work finishes."""
    tasks = list(tasks); total = len(tasks); results = []; i = 0; done = 0
    inflight = set()
    with ThreadPoolExecutor(max_workers=workers) as ex:
        while (i < total and not out_of_time()) or inflight:
            while i < total and len(inflight) < workers and not out_of_time():
                inflight.add(ex.submit(fn, tasks[i])); i += 1
            if not inflight: break
            finished, inflight = wait(inflight, return_when=FIRST_COMPLETED, timeout=5)
            for fu in finished:
                done += 1
                try: res = fu.result()
                except Exception as exc: res = {"status": f"worker_exception:{exc!r}"[:200]}
                results.append(res)
                print(f"[{label} {done}/{total}] {res.get('key','')} -> {res.get('status')} {res.get('category') or ''}")
    if i < total:
        print(f"[{label}] runtime cap reached: {total - i} task(s) not started (they stay eligible for the next run).")
    return results

def backoff(k): time.sleep(BACKOFF_BASE_SEC * (2 ** k) + RNG.uniform(0, 1.5))

PREFLIGHT_HOSTS = ("www.google.com", "www.youtube.com")
def have_network():
    """If basic DNS fails we are offline: skip the network phases WITHOUT burning any retry budget."""
    for h in PREFLIGHT_HOSTS:
        try:
            socket.setdefaulttimeout(5); socket.getaddrinfo(h, 443); return True
        except OSError:
            continue
        finally:
            socket.setdefaulttimeout(None)
    return False
ONLINE = have_network()
print("Network preflight:", "online" if ONLINE else "OFFLINE - network phases will be skipped; nothing will be logged as a failure")

Network preflight: online


## 11. Instance clip extraction

Metadata frame indices are in metadata-FPS frames and are converted to the source's actual FPS (same convention as Notebook 02). Output is written to `<id>.tmp.mp4`, **fully decoded and counted**, and only then renamed. An existing *valid* clip is never touched; an existing *invalid* clip is moved to `recovery_quarantine/`, not deleted. Non-YouTube files with default boundaries (`frame_start=1`, `frame_end=-1`) are copied verbatim.

In [11]:
def extract_instance_clip(row):
    vid = row["video_id"]; out = clip_dir / f"{vid}.mp4"; tmp = clip_dir / f"{vid}.tmp.mp4"
    base = {"key": vid, "video_id": vid}
    if out.is_file():
        if audit_video(out, "clip")["valid"]:
            return base | {"status": "already_valid"}
        shutil.move(str(out), str(quarantine_dir / f"{vid}.{RUN_ID}.invalid.mp4"))
    src = Path(row["source_path"]) if row["source_path"] else None
    if src is None or not src.is_file():
        return base | {"status": "source_missing"}
    sa = cached_audit(src, "source")
    if not sa["valid"]:
        return base | {"status": "source_invalid", "category": sa["error"]}
    default_bounds = (int(row["frame_start"]) <= 1 and int(row["frame_end"]) == -1)
    if (not row["is_youtube"]) and default_bounds and src.suffix.lower() == ".mp4":
        shutil.copy2(src, tmp)
        a = audit_video(tmp, "clip", full_decode=True)
        if a["valid"]:
            os.replace(tmp, out); return base | {"status": "success", "method": "copy_whole_file"}
        tmp.unlink(missing_ok=True); return base | {"status": "invalid_after_copy", "category": a["error"]}
    cap = cv2.VideoCapture(str(src))
    try:
        sfps, total = float(cap.get(cv2.CAP_PROP_FPS)), int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        mfps = float(row["meta_fps"])
        if sfps <= 0 or mfps <= 0 or total <= 0:
            return base | {"status": "invalid_source_metadata"}
        start = max(0, round((int(row["frame_start"]) - 1) * sfps / mfps))
        end = total - 1 if int(row["frame_end"]) == -1 else min(total - 1, round(int(row["frame_end"]) * sfps / mfps) - 1)
        if start >= total or end < start:
            return base | {"status": "invalid_frame_range", "category": f"start={start},end={end},total={total}"}
        w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        wr = cv2.VideoWriter(str(tmp), cv2.VideoWriter_fourcc(*"mp4v"), sfps, (w, h))
        if not wr.isOpened():
            wr.release(); return base | {"status": "writer_open_failed"}
        cap.set(cv2.CAP_PROP_POS_FRAMES, start)
        expected, written = end - start + 1, 0
        try:
            for _ in range(expected):
                ok, fr = cap.read()
                if not ok: break
                wr.write(fr); written += 1
        finally:
            wr.release()
        a = audit_video(tmp, "clip", expected_frames=expected, full_decode=True)
        if written == expected and a["valid"]:
            os.replace(tmp, out)
            return base | {"status": "success", "method": "crop", "frames": written}
        tmp.unlink(missing_ok=True)
        return base | {"status": "incomplete_or_invalid", "category": a["error"] or f"written={written}/{expected}"}
    finally:
        cap.release()

def run_extraction():
    a = build_audit()
    todo = a[(a.acq_class == "source_valid_clip_pending") | ((a.source_valid) & (~a.clip_valid))]
    # train first, then val, then test
    todo = todo.sort_values(by="split", key=lambda s: s.map({"train": 0, "val": 1, "test": 2}))
    rows = [r for r in todo.to_dict("records")]
    print(f"Extraction candidates: {len(rows)}")
    res = run_bounded(rows, extract_instance_clip, EXTRACT_WORKERS, "extract")
    for r in res:
        if r.get("status") not in ("already_valid",):
            log_attempt(kind="clip_extract", source_key=r.get("key"), video_ids=r.get("video_id"), status=r.get("status"),
                        category=r.get("category"), permanence=("success" if r.get("status") == "success" else "temporary"),
                        detail=r.get("method"))
    st = Counter(r.get("status") for r in res)
    print("Extraction results:", dict(st))
    return res

extraction_results = run_extraction() if RUN_EXTRACTION else print("Extraction OFF") or []

Extraction candidates: 104
[extract 1/104] 21872 -> success 
[extract 2/104] 14888 -> success 
[extract 3/104] 14887 -> success 
[extract 4/104] 14886 -> success 
[extract 5/104] 69302 -> success 
[extract 6/104] 13636 -> success 
[extract 7/104] 14884 -> success 
[extract 8/104] 21874 -> success 
[extract 9/104] 13635 -> success 
[extract 10/104] 13633 -> success 
[extract 11/104] 64283 -> success 
[extract 12/104] 08920 -> success 
[extract 13/104] 08919 -> success 
[extract 14/104] 63232 -> success 
[extract 15/104] 63231 -> success 
[extract 16/104] 13634 -> success 
[extract 17/104] 63229 -> success 
[extract 18/104] 27208 -> success 
[extract 19/104] 64288 -> success 
[extract 20/104] 64280 -> success 
[extract 21/104] 69364 -> success 
[extract 22/104] 64287 -> success 
[extract 23/104] 62164 -> success 
[extract 24/104] 64213 -> success 
[extract 25/104] 57943 -> success 
[extract 26/104] 57942 -> success 
[extract 27/104] 27207 -> success 
[extract 28/104] 57937 -> success 
[e

## 12. YouTube recovery

Each unique video id is downloaded once into `recovery_staging/`, validated with OpenCV, then moved into `data/raw/youtube_sources/` only if no file for that id exists yet. yt-dlp's own return code is never trusted.

In [12]:
def yt_format_ladder():
    h = MAX_HEIGHT
    return [f"bv*[height<={h}][vcodec^=avc1]/bv*[height<={h}]/b[height<={h}]/bv*/b", "b[ext=mp4]/b"]

class _QuietLogger:
    def __init__(self): self.msgs = []
    def debug(self, m): pass
    def info(self, m): pass
    def warning(self, m): pass
    def error(self, m): self.msgs.append(str(m))

def download_youtube_source(sid):
    import yt_dlp
    t0 = time.time(); key = sid
    existing = find_video_file(source_dir, sid)
    if existing and audit_video(existing)["valid"]:
        return {"key": key, "status": "already_valid", "category": None}
    url = f"https://www.youtube.com/watch?v={sid}"
    last_cat, last_perm, last_msg = "other_error", "temporary", ""
    ladder = yt_format_ladder(); fmt_i = 0; tries = 0
    while tries < IN_RUN_TRIES and not out_of_time():
        logger = _QuietLogger()
        opts = {"format": ladder[fmt_i], "outtmpl": str(staging_dir / "%(id)s.%(ext)s"),
                "merge_output_format": "mp4", "noplaylist": True, "quiet": True, "no_warnings": True,
                "noprogress": True, "logger": logger, "overwrites": False,
                "socket_timeout": YT_SOCKET_TIMEOUT_SEC, "retries": 2, "fragment_retries": 2, "extractor_retries": 1,
                "sleep_interval": 1, "max_sleep_interval": 3, "concurrent_fragment_downloads": 2,
                "js_runtimes": {"node": {}}, "remote_components": {"ejs:github"}}
        if COOKIES_FROM_BROWSER:
            opts["cookiesfrombrowser"] = tuple(COOKIES_FROM_BROWSER)
        try:
            with yt_dlp.YoutubeDL(opts) as ydl:
                ydl.download([url])
            files = [q for q in staging_dir.glob(f"{sid}.*") if q.suffix not in (".part", ".ytdl") and ".f" not in q.stem[-6:]]
            if not files:
                raise RuntimeError("incomplete: yt-dlp finished but produced no file")
            f = max(files, key=lambda q: q.stat().st_size)
            a = audit_video(f, "source")
            if a["valid"]:
                dest = source_dir / f"{sid}{f.suffix.lower()}"
                if find_video_file(source_dir, sid):          # duplicate prevention
                    f.unlink(missing_ok=True)
                    return {"key": key, "status": "already_valid", "category": "duplicate_prevented"}
                shutil.move(str(f), str(dest))
                for junk in staging_dir.glob(f"{sid}.*"): junk.unlink(missing_ok=True)
                log_attempt(kind="source_youtube", source_key=key, video_ids=sid, status="success", category="ok",
                            permanence="success", detail=f"{a['width']}x{a['height']}@{a['fps']:.2f} fmt#{fmt_i}",
                            bytes=dest.stat().st_size, seconds=round(time.time() - t0, 1))
                return {"key": key, "status": "success", "category": None}
            # downloaded but undecodable (e.g. AV1/VP9 build issues): drop, try the next format once
            f.unlink(missing_ok=True)
            last_cat, last_perm, last_msg = "downloaded_but_invalid:" + str(a["error"]), "temporary", str(a["error"])
            if fmt_i + 1 < len(ladder): fmt_i += 1
        except Exception as exc:
            last_msg = " | ".join(logger.msgs[-2:]) or str(exc)
            last_cat, last_perm = classify_text(last_msg)
            if last_perm == "permanent": break                 # do not retry dead videos
            if last_cat == "format_unavailable" and fmt_i + 1 < len(ladder): fmt_i += 1
            elif last_cat in ("rate_limited", "bot_check"): backoff(tries + 1)
        tries += 1
        if tries < IN_RUN_TRIES: backoff(tries - 1)
    for junk in staging_dir.glob(f"{sid}.*"): junk.unlink(missing_ok=True)
    log_attempt(kind="source_youtube", source_key=key, video_ids=sid, status="failed", category=last_cat,
                permanence=last_perm, detail=last_msg[:300], seconds=round(time.time() - t0, 1))
    return {"key": key, "status": "failed", "category": f"{last_cat}/{last_perm}"}

yt_results = []
if RUN_YOUTUBE and not ONLINE:
    print("YouTube recovery skipped: offline.")
elif RUN_YOUTUBE:
    ids = priority0[priority0.is_youtube].source_key.tolist()
    if LIMIT_NEW_YT_SOURCES: ids = ids[:LIMIT_NEW_YT_SOURCES]
    print(f"YouTube: {len(ids)} eligible sources queued (priority order).")
    yt_results = run_bounded(ids, download_youtube_source, YT_WORKERS, "yt")
    print("YouTube results:", dict(Counter(r.get("status") for r in yt_results)))
    print("Failure categories:", dict(Counter(r.get("category") for r in yt_results if r.get("status") == "failed")))
else:
    print("YouTube recovery OFF.")

YouTube: 172 eligible sources queued (priority order).
[yt 1/172] ZxOqr6aCVLc -> failed unavailable/permanent
[yt 2/172] pupBwmtznhI -> failed unavailable/permanent
[yt 3/172] qMgcmGOiNmw -> failed unavailable/permanent
[yt 4/172] rDAGt54vEsI -> failed unavailable/permanent
[yt 5/172] HAqSIA2GG3g -> failed unavailable/permanent
[yt 6/172] HNp5UcfuRSI -> failed unavailable/permanent
[yt 7/172] JxtYMfvnp58 -> failed unavailable/permanent
[yt 8/172] I0v6P6HEGCA -> failed unavailable/permanent
[yt 9/172] M1gXuGSWqVI -> failed unavailable/permanent
[yt 10/172] Kw9nFq7HD4Y -> failed unavailable/permanent
[yt 11/172] Mh3UZ1UsXzw -> failed unavailable/permanent
[yt 12/172] My4w56H6qwk -> failed unavailable/permanent
[yt 13/172] OQM3x-HY2dM -> failed unavailable/permanent
[yt 14/172] NGNoEmYEpCQ -> failed unavailable/permanent
[yt 15/172] OgcjUatW9LA -> failed unavailable/permanent
[yt 16/172] QPiZO8Aswgo -> failed unavailable/permanent
[yt 17/172] SbvomlBCAe8 -> failed unavailable/permanent
[y

## 13. Non-YouTube recovery

Content is sniffed from magic bytes. HTML is never saved as video; `.swf` is stored as `<id>.swf` (official convention) and optionally converted; 404/410/401/403 are permanent and never retried; the request carries no credentials and nothing here attempts to get around access controls. One download per unique URL is fanned out to every instance id that uses it.

In [13]:
_host_sem, _host_last, _host_fail, _host_ok = defaultdict(lambda: threading.Semaphore(PER_HOST_MAX)), {}, Counter(), Counter()
_host_lock = threading.Lock()

def sniff(head):
    h = head[:16]
    if h[:3] in (b"FWS", b"CWS", b"ZWS"): return ".swf"
    if h[4:8] == b"ftyp": return ".mp4"
    if h[:4] == b"RIFF" and head[8:12] == b"AVI ": return ".avi"
    if h[:4] == b"\x1a\x45\xdf\xa3": return ".webm"
    if h[:3] == b"FLV": return ".flv"
    low = head[:512].lower().lstrip()
    if low.startswith(b"<") or b"<html" in low or b"<!doctype" in low: return "html"
    return None

def fetch_url(url, dest):
    host = urlparse(url).netloc.lower()
    headers = {"User-Agent": UA, "Accept": "*/*"}
    if "aslpro" in host: headers["Referer"] = "http://www.aslpro.com/cgi-bin/aslpro/aslpro.cgi"   # as in the official downloader
    with _host_sem[host]:
        with _host_lock:
            wait_for = HOST_MIN_DELAY_SEC - (time.time() - _host_last.get(host, 0))
        if wait_for > 0: time.sleep(wait_for)
        try:
            req = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(req, timeout=HTTP_TIMEOUT_SEC) as resp:
                code, ctype, final = resp.status, resp.headers.get_content_type(), resp.geturl()
                first = resp.read(1024)
                kind = sniff(first)
                if kind == "html": return {"ok": False, "category": "html_not_video", "permanence": "permanent",
                                           "detail": f"HTML page returned (content-type {ctype}, final url {final})", "http": code}
                if kind is None: return {"ok": False, "category": "unrecognised_content", "permanence": "permanent",
                                         "detail": f"unknown magic bytes {first[:8]!r}, content-type {ctype}", "http": code}
                t0, total = time.time(), len(first)
                with open(dest, "wb") as f:
                    f.write(first)
                    while True:
                        if time.time() - t0 > HTTP_MAX_SECONDS: raise TimeoutError("max download seconds exceeded")
                        chunk = resp.read(1 << 17)
                        if not chunk: break
                        total += len(chunk); f.write(chunk)
                        if total > MAX_DOWNLOAD_MB * 1048576: raise RuntimeError("download exceeds size cap")
                return {"ok": True, "ext": kind, "bytes": total, "http": code, "detail": f"{ctype} -> {final}"}
        except urllib.error.HTTPError as e:
            cat, perm = classify_http(e.code)
            return {"ok": False, "category": cat, "permanence": perm, "detail": f"HTTP {e.code} {e.reason}", "http": e.code}
        except Exception as e:
            cat, perm = classify_text(repr(e) + " " + str(getattr(e, "reason", "")))
            return {"ok": False, "category": cat, "permanence": perm, "detail": repr(e)[:250], "http": None}
        finally:
            with _host_lock: _host_last[host] = time.time()

def convert_swf(swf_path, out_mp4):
    if not shutil.which("ffmpeg"):
        return False, "ffmpeg_not_found"
    tmp = Path(str(out_mp4) + ".tmp.mp4")
    try:
        subprocess.run(["ffmpeg", "-loglevel", "error", "-y", "-i", str(swf_path), "-an",
                        "-vf", "pad=width=ceil(iw/2)*2:height=ceil(ih/2)*2", "-c:v", "libx264", "-pix_fmt", "yuv420p", str(tmp)],
                       check=True, timeout=180, capture_output=True)
    except Exception as e:
        tmp.unlink(missing_ok=True); return False, f"swf_conversion_failed:{str(e)[:120]}"
    a = audit_video(tmp, "source")
    if a["valid"]:
        os.replace(tmp, out_mp4); return True, "ok"
    tmp.unlink(missing_ok=True); return False, f"swf_output_invalid:{a['error']}"

def download_url_source(task):
    url, vids = task["url"], task["video_ids"]; host = urlparse(url).netloc.lower(); key = task["source_key"]
    t0 = time.time()
    with _host_lock:
        if _host_fail[host] >= HOST_BREAKER and _host_ok[host] == 0:
            return {"key": key, "status": "skipped_host_breaker", "category": host}
    res = None
    for k in range(IN_RUN_TRIES):
        stage = staging_dir / f"{key.replace(':','_')}.download"
        res = fetch_url(url, stage)
        if res["ok"] or res["permanence"] == "permanent" or out_of_time(): break
        if k + 1 < IN_RUN_TRIES: backoff(k)
    if not res["ok"]:
        Path(staging_dir / f"{key.replace(':','_')}.download").unlink(missing_ok=True)
        with _host_lock: _host_fail[host] += 1
        log_attempt(kind="source_url", source_key=key, video_ids=",".join(vids), status="failed", category=res["category"],
                    permanence=res["permanence"], http_status=res.get("http"), detail=res["detail"], seconds=round(time.time() - t0, 1))
        return {"key": key, "status": "failed", "category": f"{res['category']}/{res['permanence']}"}
    with _host_lock: _host_ok[host] += 1
    ext = res["ext"]; stage = staging_dir / f"{key.replace(':','_')}.download"
    ok_vids = []; final_cat = None; final_detail = None
    if ext == ".swf":
        for v in vids:
            dst_swf = non_yt_dir / f"{v}.swf"
            if not dst_swf.exists(): shutil.copy2(stage, dst_swf)
            if RUN_SWF_CONVERSION and not find_video_file(non_yt_dir, v):
                good, why = convert_swf(dst_swf, non_yt_dir / f"{v}.mp4")
                if good: ok_vids.append(v)
                else: final_cat, final_detail = why.split(":", 1)[0], why
    else:
        norm = non_yt_dir / f"__probe_{key.replace(':','_')}{ext}"
        shutil.copy2(stage, norm)
        a = audit_video(norm, "source")
        norm.unlink(missing_ok=True)
        if a["valid"]:
            for v in vids:
                dst = non_yt_dir / f"{v}{ext}"
                if find_video_file(non_yt_dir, v): continue          # never overwrite
                shutil.copy2(stage, dst); ok_vids.append(v)
        else:
            final_cat = "downloaded_but_invalid"; final_detail = f"OpenCV validation: {a['error']}"
    stage.unlink(missing_ok=True)
    if ok_vids:
        log_attempt(kind="source_url", source_key=key, video_ids=",".join(ok_vids), status="success", category="ok",
                    permanence="success", http_status=res.get("http"), detail=res["detail"], bytes=res["bytes"], seconds=round(time.time() - t0, 1))
        return {"key": key, "status": "success", "category": None}
    perm = "permanent" if (final_cat or "").startswith(("swf_conversion_failed", "swf_output_invalid")) else "temporary"
    log_attempt(kind="source_url", source_key=key, video_ids=",".join(vids), status="failed", category=final_cat or "unknown",
                permanence=perm, http_status=res.get("http"), detail=(final_detail or final_cat), bytes=res["bytes"], seconds=round(time.time() - t0, 1))
    return {"key": key, "status": "failed", "category": f"{final_cat}/{perm}"}

url_results = []
if RUN_NON_YOUTUBE and not ONLINE:
    print("Non-YouTube recovery skipped: offline.")
elif RUN_NON_YOUTUBE:
    pu = priority0[~priority0.is_youtube]
    tasks = []
    for r in pu.itertuples(index=False):
        sub = cand0[(cand0.source_key == r.source_key) & cand0.eligible]
        tasks.append({"source_key": r.source_key, "url": sub.url.iloc[0], "video_ids": sub.video_id.tolist()})
    if LIMIT_NEW_URLS: tasks = tasks[:LIMIT_NEW_URLS]
    print(f"Non-YouTube: {len(tasks)} eligible unique URLs queued (priority order).")
    url_results = run_bounded(tasks, download_url_source, HTTP_WORKERS, "http")
    print("Non-YouTube results:", dict(Counter(r.get("status") for r in url_results)))
    print("Failure categories:", dict(Counter(r.get("category") for r in url_results if r.get("status") == "failed")))
else:
    print("Non-YouTube recovery OFF.")

Non-YouTube: 1233 eligible unique URLs queued (priority order).
[http 1/1233] url:003b0156c776 -> failed dns/temporary
[http 2/1233] url:07a2cee089ce -> failed dns/temporary
[http 3/1233] url:080f5d7a9c9d -> success 
[http 4/1233] url:0a1b9ed5a075 -> failed dns/temporary
[http 5/1233] url:119b79d2406f -> success 
[http 6/1233] url:12d1d914dc0d -> success 
[http 7/1233] url:18cd0b6ee36f -> failed dns/temporary
[http 8/1233] url:183d352f3808 -> failed ssl_error/temporary
[http 9/1233] url:190f2f687086 -> failed dns/temporary
[http 10/1233] url:251dc6df775c -> success 
[http 11/1233] url:284f900fa0e0 -> success 
[http 12/1233] url:35f1aa082ef6 -> success 
[http 13/1233] url:323eccf1ecc5 -> success 
[http 14/1233] url:457786516b6f -> failed http_not_found/permanent
[http 15/1233] url:5a98e9ebe01d -> failed http_not_found/permanent
[http 16/1233] url:679f9224169c -> success 
[http 17/1233] url:8024d235ac25 -> failed dns/temporary
[http 18/1233] url:7fb890988a66 -> success 
[http 19/1233] ur

## 14. Extract clips for everything newly downloaded
Re-runs the same validated extraction (it only touches instances whose source is valid and clip is missing).

In [14]:
extraction_results2 = run_extraction() if (RUN_EXTRACTION and (yt_results or url_results)) else print("No new sources; nothing to extract.") or []

Extraction candidates: 462
[extract 1/462] 35513 -> success 
[extract 2/462] 69411 -> success 
[extract 3/462] 35512 -> success 
[extract 4/462] 32250 -> success 
[extract 5/462] 35511 -> success 
[extract 6/462] 32249 -> success 
[extract 7/462] 24641 -> success 
[extract 8/462] 32248 -> success 
[extract 9/462] 24640 -> success 
[extract 10/462] 69430 -> success 
[extract 11/462] 22955 -> success 
[extract 12/462] 69343 -> success 
[extract 13/462] 22954 -> success 
[extract 14/462] 19257 -> success 
[extract 15/462] 19259 -> success 
[extract 16/462] 19261 -> success 
[extract 17/462] 18323 -> success 
[extract 18/462] 18325 -> success 
[extract 19/462] 69307 -> success 
[extract 20/462] 22953 -> success 
[extract 21/462] 42831 -> success 
[extract 22/462] 42832 -> success 
[extract 23/462] 42833 -> success 
[extract 24/462] 51224 -> success 
[extract 25/462] 51223 -> success 
[extract 26/462] 50041 -> success 
[extract 27/462] 50040 -> success 
[extract 28/462] 50037 -> success 
[e

## 15. Final audit and recovery artifacts
Everything below is rebuilt from the files on disk. `recovery_*` files are written under `data/processed/WLASL100/`; Notebook 02's original audit files are not touched.

In [15]:
final = build_audit()
final["baseline_clip_valid"] = final.video_id.isin(baseline_valid_clips)
final["baseline_source_backed"] = final.baseline_clip_valid | final.video_id.isin(
    inst[inst.youtube_id.isin(baseline["baseline_valid_youtube_sources"])].video_id)
final["newly_recovered_clip"] = final.clip_valid & ~final.baseline_clip_valid
final["status_summary"] = np.select(
    [final.clip_valid & final.baseline_clip_valid, final.newly_recovered_clip, final.source_valid & ~final.clip_valid],
    ["usable_baseline", "usable_recovered", "source_ok_clip_failed"],
    default=final.acq_class)

# ---- instance status
final.drop(columns=["frame_start", "frame_end", "meta_fps"]).to_csv(OUT["recovery_instance_status.csv"], index=False)

# ---- source status
attempts = load_attempts()
att_n = attempts[attempts.kind.isin(["source_youtube", "source_url"])].groupby("source_key").size()
src_rows = []
for key, g in final.groupby("source_key"):
    is_yt = bool(g.is_youtube.iloc[0])
    vs = bool(g.source_valid.any())
    cats = g.fail_category.dropna()
    src_rows.append({"source_kind": "youtube" if is_yt else "url", "source_key": key, "host": g.host.iloc[0],
                     "url": g.url.iloc[0], "instances": len(g), "train": int((g.split == "train").sum()),
                     "val": int((g.split == "val").sum()), "test": int((g.split == "test").sum()),
                     "source_valid": vs, "source_newly_valid": bool(vs and key not in set(baseline["baseline_valid_youtube_sources"]) and is_yt),
                     "clips_valid": int(g.clip_valid.sum()), "attempts_in_recovery_log": int(att_n.get(key, 0)),
                     "permanence": g.permanence.dropna().iloc[0] if g.permanence.notna().any() else None,
                     "fail_category": cats.iloc[0] if len(cats) else None,
                     "fail_detail": g.fail_detail.iloc[0] if (g.fail_detail != "").any() else None,
                     "source_path": g.source_path.dropna().iloc[0] if g.source_path.notna().any() else None})
source_status = pd.DataFrame(src_rows)
source_status.loc[source_status.source_valid, ["permanence", "fail_category"]] = [None, None]
source_status.to_csv(OUT["recovery_source_status.csv"], index=False)

# ---- split availability
def split_table(flag, tag):
    t = (final.groupby("split").agg(total=("video_id", "count"), usable=(flag, "sum"),
                                    classes_represented=("gloss", lambda s: s[final.loc[s.index, flag]].nunique()),
                                    signers_represented=("signer_id", lambda s: s[final.loc[s.index, flag]].nunique()),
                                    signers_in_metadata=("signer_id", "nunique")))
    t["coverage_pct"] = (100 * t.usable / t.total).round(2)
    return t.add_prefix(tag + "_")
spl = split_table("baseline_clip_valid", "baseline").join(split_table("clip_valid", "final")).reset_index()
spl["total"] = spl.final_total
spl = spl.drop(columns=["baseline_total", "final_total"])
spl["added"] = spl.final_usable - spl.baseline_usable
src_back = final.assign(sb=final.clip_valid | final.source_valid).groupby("split").sb.sum()
spl["final_source_backed"] = spl.split.map(src_back).astype(int)
spl["final_source_backed_pct"] = (100 * spl.final_source_backed / spl.total).round(2)
spl = spl.set_index("split").loc[["train", "val", "test"]].reset_index()
spl.to_csv(OUT["recovery_split_availability.csv"], index=False)

# ---- class coverage
cc = []
for (g_, s_), g in final.groupby(["gloss", "split"]):
    cc.append({"gloss": g_, "split": s_, "total": len(g), "baseline_usable": int(g.baseline_clip_valid.sum()),
               "final_usable": int(g.clip_valid.sum()), "added": int(g.newly_recovered_clip.sum()),
               "source_valid_clip_missing": int((g.source_valid & ~g.clip_valid).sum()),
               "unrecoverable_permanent": int((g.acq_class == "missing_or_permanently_inaccessible").sum())})
class_cov = pd.DataFrame(cc)
class_cov.to_csv(OUT["recovery_class_coverage.csv"], index=False)

# ---- consolidated audit (new file; Notebook 02's audit files untouched)
final.to_csv(OUT["consolidated_acquisition_audit.csv"], index=False)

display(spl)
zero = {s: sorted(class_cov[(class_cov.split == s) & (class_cov.final_usable == 0)].gloss) for s in ("train", "val", "test")}
for s, v in zero.items(): print(f"{s}: {len(v)} classes with zero usable clips" + (f" -> {v}" if v else ""))

,split,baseline_usable,baseline_classes_represented,baseline_signers_represented,baseline_signers_in_metadata,baseline_coverage_pct,final_usable,final_classes_represented,final_signers_represented,final_signers_in_metadata,final_coverage_pct,total,added,final_source_backed,final_source_backed_pct
0,train,182,95,9,62,18.15,535,100,31,62,53.34,1003,353,535,53.34
1,val,31,29,2,15,7.31,171,91,7,15,40.33,424,140,171,40.33
2,test,244,97,2,20,39.93,317,98,12,20,51.88,611,73,317,51.88


train: 0 classes with zero usable clips
val: 9 classes with zero usable clips -> ['but', 'cheat', 'city', 'cool', 'corn', 'last', 'letter', 'meet', 'pull']
test: 2 classes with zero usable clips -> ['help', 'short']


## 16. Integrity checks
Fails loudly if any frozen input changed, any baseline clip changed, any duplicate source exists, or the audit disagrees with the disk.

In [16]:
checks = {}
cur_hashes = {str(q.relative_to(project_dir)): sha256_file(q) for q in (official_json, filtered_json, *split_files.values())}
checks["frozen_inputs_unchanged (metadata, filtered json, split CSVs, signer assignments)"] = cur_hashes == baseline["frozen_hashes"]
bad = [v for v, meta in baseline["baseline_valid_clips"].items()
       if not (clip_dir / f"{v}.mp4").is_file() or (clip_dir / f"{v}.mp4").stat().st_size != meta["size"]
       or sha256_file(clip_dir / f"{v}.mp4") != meta["sha256"]]
checks[f"baseline_valid_clips_intact ({len(baseline_valid_clips)} checked, {len(bad)} changed)"] = not bad
yt_files = Counter(p.stem for p in source_dir.iterdir() if p.suffix.lower() in VIDEO_EXTS)
dups = [k for k, c in yt_files.items() if c > 1]
checks[f"no_duplicate_youtube_source_files ({len(dups)} duplicates)"] = not dups
leftover = list(clip_dir.glob("*.tmp.mp4")) + list(staging_dir.glob("*.download"))
checks[f"no_partial_files_left ({len(leftover)})"] = not leftover
disk_valid = {v for v in final.video_id if (clip_dir / f"{v}.mp4").is_file() and audit_video(clip_dir / f"{v}.mp4", "clip")["valid"]}
checks["audit_matches_disk (clip validity re-derived from files)"] = disk_valid == set(final.loc[final.clip_valid, "video_id"])
new_ids = final.loc[final.newly_recovered_clip, "video_id"].tolist()
new_bad = [v for v in new_ids if not audit_video(clip_dir / f"{v}.mp4", "clip", full_decode=True)["valid"]]
checks[f"every_new_clip_revalidated ({len(new_ids)} new, {len(new_bad)} failed)"] = not new_bad
checks["split_assignments_unchanged_in_outputs"] = final.set_index("video_id")["split"].equals(split_df.set_index("video_id")["split"].reindex(final.set_index("video_id").index))
checks["signer_assignments_unchanged_in_outputs"] = (final.set_index("video_id")["signer_id"].astype(int)
        .equals(inst.set_index("video_id")["signer_id"].astype(int).reindex(final.set_index("video_id").index)))
# stricter-than-before re-check of old clips: report only, never modify
strict_fail = final[final.baseline_clip_valid & ~final.clip_valid].video_id.tolist()
checks[f"baseline_clips_pass_current_strict_validation ({len(strict_fail)} flagged; files not touched)"] = not strict_fail
display(pd.Series(checks, name="PASS").to_frame())

,PASS
"frozen_inputs_unchanged (metadata, filtered json, split CSVs, signer assignments)",True
"baseline_valid_clips_intact (457 checked, 0 changed)",True
no_duplicate_youtube_source_files (0 duplicates),True
no_partial_files_left (0),True
audit_matches_disk (clip validity re-derived from files),True
"every_new_clip_revalidated (566 new, 0 failed)",True
split_assignments_unchanged_in_outputs,True
signer_assignments_unchanged_in_outputs,True
baseline_clips_pass_current_strict_validation (0 flagged; files not touched),True


## 17. Summary JSON and final report

In [17]:
yt_src = source_status[source_status.source_kind == "youtube"]
url_src = source_status[source_status.source_kind == "url"]
perm_src = source_status[(~source_status.source_valid) & (source_status.permanence == "permanent")]
retry_src = source_status[(~source_status.source_valid) & (source_status.permanence == "temporary")]
unknown_src = source_status[(~source_status.source_valid) & (source_status.permanence == "unknown")]
never_src = source_status[(~source_status.source_valid) & source_status.permanence.isna()]

summary = {
    "run_id": RUN_ID, "generated_utc": utc_now(),
    "definitions": {"source_download": "a valid raw source video file on disk",
                    "usable_instance": "a valid clips/<video_id>.mp4 passing OpenCV + plausibility + frame-count validation"},
    "baseline": {"usable_instance_clips": len(baseline_valid_clips),
                 "valid_youtube_source_files": len(baseline["baseline_valid_youtube_sources"])},
    "additional_valid_youtube_sources_downloaded": int(sum(1 for r in yt_results if r.get("status") == "success")),
    "additional_valid_non_youtube_sources_downloaded": int(sum(1 for r in url_results if r.get("status") == "success")),
    "additional_usable_instances_recovered": int(final.newly_recovered_clip.sum()),
    "total_final_usable_instances": int(final.clip_valid.sum()),
    "final_coverage_pct": round(100 * final.clip_valid.mean(), 2),
    "split_availability": spl.to_dict(orient="records"),
    "zero_usable_classes": zero,
    "valid_youtube_sources_now": int(yt_src.source_valid.sum()), "youtube_sources_total": int(len(yt_src)),
    "valid_non_youtube_urls_now": int(url_src.source_valid.sum()), "non_youtube_urls_total": int(len(url_src)),
    "permanently_unavailable_sources": int(len(perm_src)),
    "permanent_failure_categories": perm_src.fail_category.value_counts().to_dict(),
    "remaining_retryable_sources": int(len(retry_src)),
    "retryable_failure_categories": retry_src.fail_category.value_counts().to_dict(),
    "unknown_reason_sources_not_yet_rechecked": int(len(unknown_src)),
    "never_attempted_sources": int(len(never_src)),
    "yt_run_results": dict(Counter(r.get("status") for r in yt_results)),
    "url_run_results": dict(Counter(r.get("status") for r in url_results)),
    "extraction_run_results": dict(Counter(r.get("status") for r in (extraction_results + extraction_results2))),
    "extraction_failure_categories": dict(Counter(r.get("category") for r in (extraction_results + extraction_results2) if r.get("status") not in ("success", "already_valid"))),
    "runtime_cap_hit": bool(out_of_time()),
    "integrity_checks": {k: bool(v) for k, v in checks.items()},
    "integrity_all_pass": bool(all(checks.values())),
}
OUT["recovery_summary.json"].write_text(json.dumps(summary, indent=2, default=str))

print(f"Additional valid YouTube sources downloaded : {summary['additional_valid_youtube_sources_downloaded']}")
print(f"Additional valid non-YouTube sources        : {summary['additional_valid_non_youtube_sources_downloaded']}")
print(f"Additional usable instance clips recovered  : {summary['additional_usable_instances_recovered']}")
print(f"Total usable instance clips                 : {summary['total_final_usable_instances']} / 2038 ({summary['final_coverage_pct']}%)")
print(f"Permanently unavailable sources             : {summary['permanently_unavailable_sources']}  {summary['permanent_failure_categories']}")
print(f"Remaining retryable sources                 : {summary['remaining_retryable_sources']}  {summary['retryable_failure_categories']}")
print(f"Integrity: {'ALL PASS' if summary['integrity_all_pass'] else 'FAILURES - see table above'}")
print("\nFiles written:")
for n, p in OUT.items(): print(" -", p)
if not summary["integrity_all_pass"]:
    raise AssertionError("Integrity checks failed - do not use these artifacts until resolved.")

Additional valid YouTube sources downloaded : 0
Additional valid non-YouTube sources        : 462
Additional usable instance clips recovered  : 566
Total usable instance clips                 : 1023 / 2038 (50.2%)
Permanently unavailable sources             : 248  {'unavailable': 136, 'private': 96, 'http_not_found': 16}
Remaining retryable sources                 : 27  {'timeout': 11, 'ssl_error': 8, 'dns': 8}
Integrity: ALL PASS

Files written:
 - /Users/sahilchauhan/sign-language-recognition/data/processed/WLASL100/recovery_attempt_log.csv
 - /Users/sahilchauhan/sign-language-recognition/data/processed/WLASL100/recovery_instance_status.csv
 - /Users/sahilchauhan/sign-language-recognition/data/processed/WLASL100/recovery_source_status.csv
 - /Users/sahilchauhan/sign-language-recognition/data/processed/WLASL100/recovery_split_availability.csv
 - /Users/sahilchauhan/sign-language-recognition/data/processed/WLASL100/recovery_class_coverage.csv
 - /Users/sahilchauhan/sign-language-recogn

## 18. Handoff notes
1. Re-running is safe: permanent failures are skipped, temporary ones respect `MAX_TEMP_ATTEMPTS_TOTAL` and `COOLDOWN_HOURS`, valid files are never rewritten.
2. `recovery_baseline.json` is the fixed reference for all "added" counts. Delete it only if you intentionally want to re-baseline.
3. Notebooks 03–05 should read `clips/` filtered by `recovery_instance_status.csv::clip_valid`. Landmark/feature artifacts for newly recovered instances must be generated **incrementally** by a later step; existing ones were not touched here.
4. Sources with `permanence == "unknown"` came from the official downloader's silent failures; this notebook rechecks them once and records the real reason.